## Module 4 - Bright spots, hot spots, and yield / crop water productivity gaps using crop performance and IPA indicators

**Last update: 19-Apr-2026**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wateraccounting/WaPORIPA/blob/main/Notebooks_v1.0/Module_4_BrightHotSpots_Yield_CWP_IPA_Gaps.ipynb)

This notebook supports **two complementary approaches**:

1. **Percentile approach = relative performance** within the study area  
2. **Gap approach = performance versus target/potential** using target crop yield and target crop water productivity (cWP)

It identifies:

- **Bright spots** = better-performing areas  
- **Hot spots** = lower-performing areas

under both:

- **Crop-only analysis:** crop yield + cWP  
- **Weighted multi-indicator analysis:** crop yield + cWP + adequacy + beneficial fraction


## 1. Import packages/libraries

In [ ]:
!pip install -q rioxarray

In [ ]:

import os
import glob
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rioxarray
from google.colab import files


## 2. Upload and unzip data

In [ ]:

# Upload the zip file that contains the seasonal outputs from the previous modules.
uploaded = files.upload()


In [ ]:

# Update the zip filename if needed.
zip_path = "/content/Yield_WP_IPA.zip"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("/content")

print("Unzipped:", zip_path)


## 3. User settings

In [ ]:

# ============================================================
# USER SETTINGS
# ============================================================

# Base folders
dir_proj = os.path.split(os.getcwd())[0]
dir_data = r"/content/content/output/"      # input base directory
dir_out  = r"/content/output/"              # output base directory

# Input folders
yield_folder        = os.path.join(dir_proj, dir_data, "yield_season")
cwp_folder          = os.path.join(dir_proj, dir_data, "Cwp_season")
adequacy_folder     = os.path.join(dir_proj, dir_data, "Adequacy")
beneficial_folder   = os.path.join(dir_proj, dir_data, "BeneficialFraction")

# Percentiles
bright_percentile = 90     # upper percentile for bright spots / default targets
hot_percentile    = 10     # lower percentile for hot spots

# --- Target settings for gap analysis ---
# target_mode:
#   "percentile" -> derive targets from the upper percentile of each raster
#   "manual"     -> use the manual values below
target_mode = "percentile"

# Manual targets (used only when target_mode = "manual")
manual_target_yield = None   # e.g. 8.0   (ton/ha or your yield unit)
manual_target_cwp   = None   # e.g. 1.50  (kg/m3 or your cWP unit)

# Gap thresholds for direct gap-based classification
bright_gap_threshold = 0.20   # <= 20% gap = bright spot candidate
hot_gap_threshold    = 0.40   # >= 40% gap = hot spot candidate

# Weighted multi-indicator analysis
# Default: equal weights
weights = {
    "yield": 1.0,
    "cwp": 1.0,
    "adequacy": 1.0,
    "beneficial_fraction": 1.0
}

# Optional strict masking of non-physical values
strict_positive_mask = False    # True -> values <= 0 are set to NaN

# Output folders
output_targets_folder                   = os.path.join(dir_out, "Targets")
output_bright_hot_crop_pct_folder       = os.path.join(dir_out, "BrightHotSpots_Crop_Percentile")
output_bright_hot_crop_gap_folder       = os.path.join(dir_out, "BrightHotSpots_Crop_Gap")
output_bright_hot_weighted_pct_folder   = os.path.join(dir_out, "BrightHotSpots_Weighted_Percentile")
output_bright_hot_weighted_gap_folder   = os.path.join(dir_out, "BrightHotSpots_Weighted_Gap")

for folder in [
    output_targets_folder,
    output_bright_hot_crop_pct_folder,
    output_bright_hot_crop_gap_folder,
    output_bright_hot_weighted_pct_folder,
    output_bright_hot_weighted_gap_folder
]:
    os.makedirs(folder, exist_ok=True)

print("Input folders:")
print(" yield               :", yield_folder)
print(" cWP                 :", cwp_folder)
print(" adequacy            :", adequacy_folder)
print(" beneficial fraction :", beneficial_folder)

print("\nTarget mode        :", target_mode)
print("Bright percentile   :", bright_percentile)
print("Hot percentile      :", hot_percentile)
print("Bright gap threshold:", bright_gap_threshold)
print("Hot gap threshold   :", hot_gap_threshold)

print("\nOutput folders:")
print(" targets                   :", output_targets_folder)
print(" crop percentile spots     :", output_bright_hot_crop_pct_folder)
print(" crop gap spots            :", output_bright_hot_crop_gap_folder)
print(" weighted percentile spots :", output_bright_hot_weighted_pct_folder)
print(" weighted gap spots        :", output_bright_hot_weighted_gap_folder)


## 4. Helper functions

In [ ]:

def sorted_tifs(folder):
    return sorted(glob.glob(os.path.join(folder, "*.tif")))

def extract_date_label(file_path):
    name = os.path.basename(file_path).split(".")[0]
    for token in [
        "pywapor_", "yield", "Cwp", "Adequacy", "BeneficialFraction",
        "Bright", "Hot", "Score", "Targets", "Gap", "Percentile",
        "Weighted", "Crop"
    ]:
        name = name.replace(token, "")
    return name.replace("__", "_").replace("_", " ").strip()

def open_clean_raster(file_path, strict_positive_mask=False):
    ds = rioxarray.open_rasterio(file_path)
    arr = ds.astype("float32")
    nodata = ds.rio.nodata
    if nodata is not None:
        arr = arr.where(arr != nodata)
    arr = arr.where(np.isfinite(arr))
    if strict_positive_mask:
        arr = arr.where(arr > 0)
    return ds, arr

def percentile_thresholds(arr, p_hot=10, p_bright=90):
    vals = arr.values
    hot = float(np.nanpercentile(vals, p_hot))
    bright = float(np.nanpercentile(vals, p_bright))
    return hot, bright

def save_array_like(template_ds, array2d, out_fh, nodata=-9999):
    out = template_ds.copy()
    out.values[0] = np.where(np.isfinite(array2d), array2d, nodata)
    out.rio.write_nodata(nodata, inplace=True)
    out.rio.to_raster(out_fh)

def plot_map(array2d, title, cmap="viridis"):
    plt.figure(figsize=(10, 6))
    plt.imshow(array2d, cmap=cmap)
    plt.colorbar(shrink=0.8)
    plt.title(title)
    plt.axis("off")
    plt.show()

def normalized_score(arr, hot_thr, bright_thr):
    denom = bright_thr - hot_thr
    if np.isclose(denom, 0):
        return np.where(np.isfinite(arr), np.nan, np.nan)
    score = (arr - hot_thr) / denom
    return np.clip(score, 0, 1)

def normalize_weights(weights_dict):
    total = sum(weights_dict.values())
    if total <= 0:
        raise ValueError("The sum of weights must be greater than zero.")
    return {k: v / total for k, v in weights_dict.items()}

def relative_gap(actual_arr, target_value):
    if target_value is None or (not np.isfinite(target_value)) or target_value <= 0:
        return np.full_like(actual_arr, np.nan, dtype="float32")
    gap = 1 - (actual_arr / target_value)
    return np.where(np.isfinite(actual_arr), np.clip(gap, 0, None), np.nan)

def gap_threshold_masks(yield_gap_arr, cwp_gap_arr, bright_gap_thr=0.20, hot_gap_thr=0.40):
    valid = np.isfinite(yield_gap_arr) & np.isfinite(cwp_gap_arr)
    bright_mask = valid & (yield_gap_arr <= bright_gap_thr) & (cwp_gap_arr <= bright_gap_thr)
    hot_mask    = valid & (yield_gap_arr >= hot_gap_thr)    & (cwp_gap_arr >= hot_gap_thr)
    return bright_mask, hot_mask

def normalize_gap_to_score(gap_arr, hot_gap_thr):
    if hot_gap_thr <= 0:
        hot_gap_thr = 1.0
    score = gap_arr / hot_gap_thr
    return np.where(np.isfinite(gap_arr), np.clip(score, 0, 1), np.nan)


## 5. Collect file handles and check availability

In [ ]:

yield_fhs      = sorted_tifs(yield_folder)
cwp_fhs        = sorted_tifs(cwp_folder)
adequacy_fhs   = sorted_tifs(adequacy_folder)
beneficial_fhs = sorted_tifs(beneficial_folder)

print("Yield files             :", len(yield_fhs))
print("cWP files               :", len(cwp_fhs))
print("Adequacy files          :", len(adequacy_fhs))
print("Beneficial fraction files:", len(beneficial_fhs))

if not yield_fhs or not cwp_fhs:
    raise FileNotFoundError("Yield and/or cWP seasonal rasters were not found.")

if len(yield_fhs) != len(cwp_fhs):
    raise ValueError("Yield and cWP file counts do not match.")

weighted_analysis_ready = (len(yield_fhs) == len(cwp_fhs) == len(adequacy_fhs) == len(beneficial_fhs))
print("Weighted multi-indicator analysis ready:", weighted_analysis_ready)


## 6. Target crop yield and target crop water productivity for gap analysis

In [ ]:

target_rows = []

for yield_fh, cwp_fh in zip(yield_fhs, cwp_fhs):
    _, yield_da = open_clean_raster(yield_fh, strict_positive_mask)
    _, cwp_da   = open_clean_raster(cwp_fh, strict_positive_mask)

    pct_target_yield = float(np.nanpercentile(yield_da.values, bright_percentile))
    pct_target_cwp   = float(np.nanpercentile(cwp_da.values, bright_percentile))
    hot_yield        = float(np.nanpercentile(yield_da.values, hot_percentile))
    hot_cwp          = float(np.nanpercentile(cwp_da.values, hot_percentile))

    target_yield = pct_target_yield if target_mode == "percentile" else manual_target_yield
    target_cwp   = pct_target_cwp   if target_mode == "percentile" else manual_target_cwp

    date_label = extract_date_label(yield_fh)

    target_rows.append({
        "Period": date_label,
        "TargetMode": target_mode,
        "TargetYield": round(float(target_yield), 2) if target_yield is not None else np.nan,
        "TargetYield_fromPctl": round(pct_target_yield, 2),
        "HotYieldThreshold": round(hot_yield, 2),
        "TargetCWP": round(float(target_cwp), 3) if target_cwp is not None else np.nan,
        "TargetCWP_fromPctl": round(pct_target_cwp, 3),
        "HotCWPThreshold": round(hot_cwp, 3)
    })

targets_df = pd.DataFrame(target_rows)
targets_df


In [ ]:

targets_csv = os.path.join(output_targets_folder, "TargetYield_TargetCWP.csv")
targets_df.to_csv(targets_csv, index=False)
print("Saved:", targets_csv)


## 7. Bright and hot spots using crop yield and crop water productivity only

In [ ]:

# ---------------------------
# 7A. Percentile approach = relative performance
# ---------------------------
crop_pct_rows = []
crop_gap_rows = []

for row, yield_fh, cwp_fh in zip(target_rows, yield_fhs, cwp_fhs):
    yield_ds, yield_da = open_clean_raster(yield_fh, strict_positive_mask)
    _,        cwp_da   = open_clean_raster(cwp_fh, strict_positive_mask)

    yield_arr = yield_da.values[0]
    cwp_arr   = cwp_da.values[0]

    yield_hot, yield_bright = percentile_thresholds(yield_da, hot_percentile, bright_percentile)
    cwp_hot, cwp_bright     = percentile_thresholds(cwp_da, hot_percentile, bright_percentile)

    valid = np.isfinite(yield_arr) & np.isfinite(cwp_arr)

    bright_mask_pct = valid & (yield_arr >= yield_bright) & (cwp_arr >= cwp_bright)
    hot_mask_pct    = valid & (yield_arr <= yield_hot)    & (cwp_arr <= cwp_hot)

    date_label = extract_date_label(yield_fh)
    safe_date  = date_label.replace(' ', '_')

    bright_out_pct = os.path.join(output_bright_hot_crop_pct_folder, f"BrightSpots_Crop_Percentile_{safe_date}.tif")
    hot_out_pct    = os.path.join(output_bright_hot_crop_pct_folder, f"HotSpots_Crop_Percentile_{safe_date}.tif")

    save_array_like(yield_ds, bright_mask_pct.astype("float32"), bright_out_pct)
    save_array_like(yield_ds, hot_mask_pct.astype("float32"), hot_out_pct)

    crop_pct_rows.append({
        "Period": date_label,
        "YieldBrightThreshold": round(yield_bright, 2),
        "YieldHotThreshold": round(yield_hot, 2),
        "CWPBrightThreshold": round(cwp_bright, 3),
        "CWPHotThreshold": round(cwp_hot, 3),
        "BrightPixels": int(np.nansum(bright_mask_pct)),
        "HotPixels": int(np.nansum(hot_mask_pct))
    })

    # ---------------------------
    # 7B. Gap approach = performance versus target / potential
    # ---------------------------
    yield_gap = relative_gap(yield_arr, row["TargetYield"])
    cwp_gap   = relative_gap(cwp_arr, row["TargetCWP"])

    bright_mask_gap, hot_mask_gap = gap_threshold_masks(
        yield_gap, cwp_gap,
        bright_gap_thr=bright_gap_threshold,
        hot_gap_thr=hot_gap_threshold
    )

    yield_gap_out = os.path.join(output_bright_hot_crop_gap_folder, f"YieldGap_{safe_date}.tif")
    cwp_gap_out   = os.path.join(output_bright_hot_crop_gap_folder, f"CWPGap_{safe_date}.tif")
    bright_out_gap = os.path.join(output_bright_hot_crop_gap_folder, f"BrightSpots_Crop_Gap_{safe_date}.tif")
    hot_out_gap    = os.path.join(output_bright_hot_crop_gap_folder, f"HotSpots_Crop_Gap_{safe_date}.tif")

    save_array_like(yield_ds, yield_gap.astype("float32"), yield_gap_out)
    save_array_like(yield_ds, cwp_gap.astype("float32"), cwp_gap_out)
    save_array_like(yield_ds, bright_mask_gap.astype("float32"), bright_out_gap)
    save_array_like(yield_ds, hot_mask_gap.astype("float32"), hot_out_gap)

    crop_gap_rows.append({
        "Period": date_label,
        "TargetYield": row["TargetYield"],
        "TargetCWP": row["TargetCWP"],
        "MeanYieldGap": round(float(np.nanmean(yield_gap)), 3),
        "MeanCWPGap": round(float(np.nanmean(cwp_gap)), 3),
        "BrightGapThreshold": bright_gap_threshold,
        "HotGapThreshold": hot_gap_threshold,
        "BrightPixels": int(np.nansum(bright_mask_gap)),
        "HotPixels": int(np.nansum(hot_mask_gap))
    })

crop_spots_percentile_df = pd.DataFrame(crop_pct_rows)
crop_spots_gap_df        = pd.DataFrame(crop_gap_rows)

print("Crop percentile-based spots")
display(crop_spots_percentile_df)

print("Crop gap-based spots")
display(crop_spots_gap_df)


## 8. Weighted bright and hot spots using yield, cWP, adequacy, and beneficial fraction

In [ ]:

if not weighted_analysis_ready:
    print("Weighted analysis skipped because one or more indicator folders are missing or the file counts do not match.")
else:
    w = normalize_weights(weights)
    print("Normalized weights:", w)

    weighted_pct_rows = []
    weighted_gap_rows = []

    for row, yield_fh, cwp_fh, adequacy_fh, beneficial_fh in zip(target_rows, yield_fhs, cwp_fhs, adequacy_fhs, beneficial_fhs):
        base_ds, yield_da = open_clean_raster(yield_fh, strict_positive_mask)
        _, cwp_da         = open_clean_raster(cwp_fh, strict_positive_mask)
        _, adequacy_da    = open_clean_raster(adequacy_fh, strict_positive_mask)
        _, beneficial_da  = open_clean_raster(beneficial_fh, strict_positive_mask)

        y_arr = yield_da.values[0]
        w_arr = cwp_da.values[0]
        a_arr = adequacy_da.values[0]
        b_arr = beneficial_da.values[0]

        valid = np.isfinite(y_arr) & np.isfinite(w_arr) & np.isfinite(a_arr) & np.isfinite(b_arr)

        # ----------------------------------------------------
        # 8A. Weighted percentile approach = relative performance
        # ----------------------------------------------------
        y_hot, y_bright = percentile_thresholds(yield_da, hot_percentile, bright_percentile)
        w_hot, w_bright = percentile_thresholds(cwp_da, hot_percentile, bright_percentile)
        a_hot, a_bright = percentile_thresholds(adequacy_da, hot_percentile, bright_percentile)
        b_hot, b_bright = percentile_thresholds(beneficial_da, hot_percentile, bright_percentile)

        y_score = normalized_score(y_arr, y_hot, y_bright)
        w_score = normalized_score(w_arr, w_hot, w_bright)
        a_score = normalized_score(a_arr, a_hot, a_bright)
        b_score = normalized_score(b_arr, b_hot, b_bright)

        valid_pct = np.isfinite(y_score) & np.isfinite(w_score) & np.isfinite(a_score) & np.isfinite(b_score)

        composite_score_pct = (
            w["yield"] * y_score +
            w["cwp"] * w_score +
            w["adequacy"] * a_score +
            w["beneficial_fraction"] * b_score
        )
        composite_score_pct = np.where(valid_pct, composite_score_pct, np.nan)

        composite_hot_pct    = float(np.nanpercentile(composite_score_pct, hot_percentile))
        composite_bright_pct = float(np.nanpercentile(composite_score_pct, bright_percentile))

        bright_mask_pct = valid_pct & (composite_score_pct >= composite_bright_pct)
        hot_mask_pct    = valid_pct & (composite_score_pct <= composite_hot_pct)

        date_label = extract_date_label(yield_fh)
        safe_date  = date_label.replace(' ', '_')

        score_out_pct  = os.path.join(output_bright_hot_weighted_pct_folder, f"CompositeScore_Percentile_{safe_date}.tif")
        bright_out_pct = os.path.join(output_bright_hot_weighted_pct_folder, f"BrightSpots_Weighted_Percentile_{safe_date}.tif")
        hot_out_pct    = os.path.join(output_bright_hot_weighted_pct_folder, f"HotSpots_Weighted_Percentile_{safe_date}.tif")

        save_array_like(base_ds, composite_score_pct.astype("float32"), score_out_pct)
        save_array_like(base_ds, bright_mask_pct.astype("float32"), bright_out_pct)
        save_array_like(base_ds, hot_mask_pct.astype("float32"), hot_out_pct)

        weighted_pct_rows.append({
            "Period": date_label,
            "CompositeBrightThreshold": round(composite_bright_pct, 3),
            "CompositeHotThreshold": round(composite_hot_pct, 3),
            "BrightPixels": int(np.nansum(bright_mask_pct)),
            "HotPixels": int(np.nansum(hot_mask_pct)),
            "WeightYield": round(w["yield"], 3),
            "WeightCWP": round(w["cwp"], 3),
            "WeightAdequacy": round(w["adequacy"], 3),
            "WeightBeneficialFraction": round(w["beneficial_fraction"], 3)
        })

        # ----------------------------------------------------
        # 8B. Weighted gap approach = performance versus target
        # ----------------------------------------------------
        yield_gap = relative_gap(y_arr, row["TargetYield"])
        cwp_gap   = relative_gap(w_arr, row["TargetCWP"])
        adequacy_gap = np.where(np.isfinite(a_arr), np.clip(1 - a_arr, 0, None), np.nan)
        beneficial_gap = np.where(np.isfinite(b_arr), np.clip(1 - b_arr, 0, None), np.nan)

        y_gap_score = normalize_gap_to_score(yield_gap, hot_gap_threshold)
        w_gap_score = normalize_gap_to_score(cwp_gap, hot_gap_threshold)
        a_gap_score = normalize_gap_to_score(adequacy_gap, hot_gap_threshold)
        b_gap_score = normalize_gap_to_score(beneficial_gap, hot_gap_threshold)

        valid_gap = np.isfinite(y_gap_score) & np.isfinite(w_gap_score) & np.isfinite(a_gap_score) & np.isfinite(b_gap_score)

        composite_gap_score = (
            w["yield"] * y_gap_score +
            w["cwp"] * w_gap_score +
            w["adequacy"] * a_gap_score +
            w["beneficial_fraction"] * b_gap_score
        )
        composite_gap_score = np.where(valid_gap, composite_gap_score, np.nan)

        bright_mask_gap = valid_gap & (composite_gap_score <= bright_gap_threshold)
        hot_mask_gap    = valid_gap & (composite_gap_score >= hot_gap_threshold)

        gap_score_out   = os.path.join(output_bright_hot_weighted_gap_folder, f"CompositeGapScore_{safe_date}.tif")
        yield_gap_out   = os.path.join(output_bright_hot_weighted_gap_folder, f"YieldGap_{safe_date}.tif")
        cwp_gap_out     = os.path.join(output_bright_hot_weighted_gap_folder, f"CWPGap_{safe_date}.tif")
        bright_out_gap  = os.path.join(output_bright_hot_weighted_gap_folder, f"BrightSpots_Weighted_Gap_{safe_date}.tif")
        hot_out_gap     = os.path.join(output_bright_hot_weighted_gap_folder, f"HotSpots_Weighted_Gap_{safe_date}.tif")

        save_array_like(base_ds, composite_gap_score.astype("float32"), gap_score_out)
        save_array_like(base_ds, yield_gap.astype("float32"), yield_gap_out)
        save_array_like(base_ds, cwp_gap.astype("float32"), cwp_gap_out)
        save_array_like(base_ds, bright_mask_gap.astype("float32"), bright_out_gap)
        save_array_like(base_ds, hot_mask_gap.astype("float32"), hot_out_gap)

        weighted_gap_rows.append({
            "Period": date_label,
            "TargetYield": row["TargetYield"],
            "TargetCWP": row["TargetCWP"],
            "MeanCompositeGapScore": round(float(np.nanmean(composite_gap_score)), 3),
            "MeanYieldGap": round(float(np.nanmean(yield_gap)), 3),
            "MeanCWPGap": round(float(np.nanmean(cwp_gap)), 3),
            "BrightGapThreshold": bright_gap_threshold,
            "HotGapThreshold": hot_gap_threshold,
            "BrightPixels": int(np.nansum(bright_mask_gap)),
            "HotPixels": int(np.nansum(hot_mask_gap)),
            "WeightYield": round(w["yield"], 3),
            "WeightCWP": round(w["cwp"], 3),
            "WeightAdequacy": round(w["adequacy"], 3),
            "WeightBeneficialFraction": round(w["beneficial_fraction"], 3)
        })

    weighted_spots_percentile_df = pd.DataFrame(weighted_pct_rows)
    weighted_spots_gap_df        = pd.DataFrame(weighted_gap_rows)

    print("Weighted percentile-based spots")
    display(weighted_spots_percentile_df)

    print("Weighted gap-based spots")
    display(weighted_spots_gap_df)


## 9. Optional quick visualization of one output

In [ ]:

# Example: plot the first weighted gap bright spot raster if it exists
example_files = sorted_tifs(output_bright_hot_weighted_gap_folder)

if example_files:
    ds = rioxarray.open_rasterio(example_files[0])
    arr = ds.values[0]
    plot_map(arr, f"Example output: {os.path.basename(example_files[0])}", cmap="RdYlGn")
else:
    print("No example output found yet.")


## 10. Download results

In [ ]:

zip_out = "/content/Module_4_BrightHotSpots_Productivity_IPA.zip"
!zip -r $zip_out /content/output/ > /dev/null
print("Created:", zip_out)
files.download(zip_out)
